<a href="https://colab.research.google.com/github/kdimbu-spec/Intelligent-Agents-Portfolio/blob/main/Individual_Implementation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This installs Gradio, Pandas, Requests, arXiv, ReportLab and pydantic, among other dependencies.

In [50]:
!pip -q install arxiv
!pip -q install requests
!pip -q install pandas
!pip -q install pydantic
!pip -q install gradio
!pip -q install groq
!pip -q install scikit-learn
!pip -q install nltk
!pip -q install reportlab
!pip -q install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

#Create Shared State
This is the memory shared between all agents.

In [3]:
from pydantic import BaseModel
from typing import List, Dict


class ResearchState(BaseModel):

    question: str

    plan: Dict = {}

    retrieved_papers: List[Dict] = []

    processed_papers: List[Dict] = []

    errors: List[str] = []

    cycle: int = 0

    sufficient: bool = False

# Create Planner Agent

The planner converts one research question into multiple search tasks.

In [4]:
def planner_agent(question):

    tasks = [
        {
            "query": question,
            "purpose": "General literature review"
        },
        {
            "query": f"{question} recent advances",
            "purpose": "Recent publications"
        },
        {
            "query": f"{question} challenges",
            "purpose": "Research gaps"
        }
    ]

    return {
        "tasks": tasks
    }

In [18]:
##TESTING
plan = planner_agent(
    "he impact of modern technology on student learning experiences in higher education in europe"
)

plan

{'tasks': [{'query': 'he impact of modern technology on student learning experiences in higher education in europe',
   'purpose': 'General literature review'},
  {'query': 'he impact of modern technology on student learning experiences in higher education in europe recent advances',
   'purpose': 'Recent publications'},
  {'query': 'he impact of modern technology on student learning experiences in higher education in europe challenges',
   'purpose': 'Research gaps'}]}

#Create arXiv Retrieval Agent

In [19]:
import arxiv

In [20]:
def search_arxiv(query, max_results=5):
    papers = []
    try:
        client = arxiv.Client()
        search = arxiv.Search(
            query=query,
            max_results=max_results
        )
        for result in client.results(search):
            papers.append({
                "title": result.title,
                "authors": [
                    author.name
                    for author in result.authors
                ],
                "summary": result.summary,
                "url": result.entry_id,
                "source": "arxiv",
                "year": result.published.year
            })
    except Exception as e:
        print(f"arXiv error: {e}")
    return papers

In [21]:
## testing the retrieval agent
papers = search_arxiv(
    "Explainable AI Healthcare"
)

print(
    f"{len(papers)} papers found"
)

if len(papers) > 0:
    print(papers[0])
else:
    print("No papers retrieved due to rate limiting or search failure. Please wait a few seconds and try executing again.")

5 papers found
{'title': 'Explainable AI for Blind and Low-Vision Users: Navigating Trust, Modality, and Interpretability in the Agentic Era', 'authors': ['Abu Noman Md Sakib', 'Protik Dey', 'Zijie Zhang', 'Taslima Akter'], 'summary': 'Explainable Artificial Intelligence (XAI) is critical for ensuring trust and accountability, yet its development remains predominantly visual. For blind and low-vision (BLV) users, the lack of accessible explanations creates a fundamental barrier to the independent use of AI-driven assistive technologies. This problem intensifies as AI systems shift from single-query tools into autonomous agents that take multi-step actions and make consequential decisions across extended task horizons, where a single undetected error can propagate irreversibly before any feedback is available. This paper investigates the unique XAI requirements of the BLV community through a comprehensive analysis of user interviews and contemporary research. By examining usage patterns

# Create OpenAlex Retrieval Agent

OpenAlex requires no API key.

In [22]:
import requests
def search_openalex(
    query,
    max_results=5
):

    url = (
        "https://api.openalex.org/works"
    )

    params = {
        "search": query,
        "per-page": max_results
    }

    papers = []

    try:

        response = requests.get(
            url,
            params=params,
            timeout=20
        )

        data = response.json()

        for work in data.get(
            "results",
            []
        ):

            papers.append({
                "title":
                work.get(
                    "display_name",
                    ""
                ),

                "summary":
                "Abstract not available",

                "doi":
                work.get(
                    "doi"
                ),

                "year":
                work.get(
                    "publication_year"
                ),

                "citations":
                work.get(
                    "cited_by_count",
                    0
                ),

                "url":
                work.get(
                    "id"
                ),

                "source":
                "openalex"
            })

    except Exception as e:

        print(
            f"OpenAlex error: {e}"
        )

    return papers

In [24]:
##Testing
papers = search_openalex(
    "The impact of modern technology on student learning experiences in higher education in europe"
)

print(
    len(papers)
)

papers[0]

5


{'title': 'Impact of online tools on the learning experience of students in higher education',
 'summary': 'Abstract not available',
 'doi': 'https://doi.org/10.1051/shsconf/202315606003',
 'year': 2023,
 'citations': 5,
 'url': 'https://openalex.org/W4320016324',
 'source': 'openalex'}

# Create Parallel Retrieval Agent

Instead of waiting for one source at a time, query them simultaneously.

In [25]:
from concurrent.futures import (
    ThreadPoolExecutor
)

def retrieve_papers(
    query
):

    papers = []

    with ThreadPoolExecutor(
        max_workers=2
    ) as executor:

        arxiv_future = executor.submit(
            search_arxiv,
            query
        )

        openalex_future = executor.submit(
            search_openalex,
            query
        )

        papers.extend(
            arxiv_future.result()
        )

        papers.extend(
            openalex_future.result()
        )

    return papers

In [26]:
papers = retrieve_papers(
    "Explainable AI Healthcare"
)

print(
    f"Retrieved {len(papers)} papers"
)

Retrieved 10 papers


# Create Deduplication Agent

Some papers appear in multiple databases.

In [27]:
def deduplicate_papers(
    papers
):

    unique = {}

    for paper in papers:

        key = paper.get(
            "title",
            ""
        ).lower()

        if key not in unique:

            unique[key] = paper

    return list(
        unique.values()
    )

In [28]:
unique_papers = (
    deduplicate_papers(
        papers
    )
)

print(
    len(unique_papers)
)

10


# Create Workflow Orchestrator

This controls the agents.

In [29]:
def run_research(
    research_question,
    sources,
    max_cycles=1
):

    state = ResearchState(
        question=research_question
    )

    state.plan = planner_agent(
        research_question
    )

    all_papers = []

    tasks = state.plan["tasks"]

    for task in tasks:

        papers = retrieve_papers(
            task["query"]
        )

        all_papers.extend(
            papers
        )

    state.retrieved_papers = (
        deduplicate_papers(
            all_papers
        )
    )

    state.cycle = 1

    state.sufficient = (
        len(
            state.retrieved_papers
        ) > 0
    )

    return state

In [30]:
state = run_research(
    research_question=
    "The impact of modern technology on student learning experiences in higher education in europe",

    sources=[
        "arxiv",
        "openalex"
    ]
)

print(
    f"Papers found: "
    f"{len(state.retrieved_papers)}"
)

Papers found: 15


# Configure Groq API

In [31]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass(
    "Enter Groq API Key: "
)

Enter Groq API Key: ··········


Create Groq Client

In [32]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
allam-2-7b
qwen/qwen3.8-27b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b


## Create Summarization Agent

Create an agent that summarizes papers consistently.

In [33]:
def summarize_paper(text):

    try:

        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": """
                    Summarize the paper in four sections:

                    1. Objective
                    2. Methodology
                    3. Key Findings
                    4. Limitations

                    Keep under 500 words.
                    """
                },
                {
                    "role": "user",
                    "content": text[:9000]
                }
            ]
        )

        return (
            response
            .choices[0]
            .message.content
        )

    except Exception as e:

        return f"Summary Error: {e}"

In [34]:
print(
    summarize_paper(
        state.retrieved_papers[0]["summary"]
    )
)

**1. Objective**  
The study aims to determine how higher‑education institutions (HEIs) can create and enhance the *experiential value* of their products and services so that student satisfaction increases. It emphasizes the role of strong, co‑created ties between universities, current students, and alumni as a pathway to higher perceived value and satisfaction.

**2. Methodology**  
- **Design:** Quantitative, survey‑based, confirmatory research.  
- **Population & Sample:** 357 undergraduate students drawn from 32 universities in South Sumatra Province, Indonesia.  
- **Sampling:** Stratified random sampling ensured representation across faculties and year levels.  
- **Instruments:** Structured questionnaire measuring (a) co‑creation activities (student involvement in campus events, decision‑making, service design), (b) experiential value of higher‑education products/services, and (c) overall student satisfaction.  
- **Data Analysis:** Structural Equation Modeling (SEM) was applied

## Create Keyword Extraction Agent

In [35]:
from collections import Counter
import re

def extract_keywords(text):

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        text.lower()
    )

    common = Counter(words)

    return [
        keyword
        for keyword, count
        in common.most_common(10)
    ]

In [36]:
extract_keywords(
    state.retrieved_papers[0]["summary"]
)

['students',
 'higher',
 'student',
 'experience',
 'values',
 'that',
 'satisfaction',
 'with',
 'value',
 'education']

## Create Relevance Scoring Agent

Instead of assigning fixed scores, calculate them.

In [37]:
from sklearn.feature_extraction.text import (
    TfidfVectorizer
)
from sklearn.metrics.pairwise import (
    cosine_similarity
)

def calculate_similarity(
    query,
    document
):

    vectorizer = TfidfVectorizer()

    vectors = vectorizer.fit_transform(
        [query, document]
    )

    score = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]

    return score

Calculate Recency Score

In [38]:
from datetime import datetime

def recency_score(year):

    current_year = datetime.now().year

    age = current_year - year

    score = max(
        0,
        1 - (age / 10)
    )

    return score

Citation Score

In [39]:
def citation_score(citations):

    max_citations = 1000

    return min(
        citations / max_citations,
        1.0
    )

Calculate Final Score

In [40]:
def evaluate_paper(
    paper,
    research_question
):

    text = (
        paper.get("title","")
        + " "
        + paper.get("summary","")
    )

    similarity = calculate_similarity(
        research_question,
        text
    )

    year = paper.get(
        "year",
        datetime.now().year
    )

    citations = paper.get(
        "citations",
        0
    )

    recency = recency_score(year)

    citation_strength = (
        citation_score(
            citations
        )
    )

    final_score = (
        0.5 * similarity +
        0.3 * recency +
        0.2 * citation_strength
    )

    return {
        "score": round(
            final_score,
            3
        ),
        "reason":
        (
            f"Similarity={similarity:.2f}, "
            f"Recency={recency:.2f}, "
            f"Citations={citation_strength:.2f}"
        )
    }

Citation Verification Agent

In [41]:
def verify_citation(
    paper
):

    checks = {

        "title":
        bool(
            paper.get("title")
        ),

        "year":
        bool(
            paper.get("year")
        ),

        "source":
        bool(
            paper.get("source")
        ),

        "url":
        bool(
            paper.get("url")
        )
    }

    return checks

## Evidence Quality Agent

Evidence quality should not rely only on citation count.

In [42]:
def assess_evidence_quality(
    paper
):

    score = 0

    if paper.get("year",0) >= 2022:
        score += 1

    if paper.get("citations",0) > 20:
        score += 1

    if paper.get("doi"):
        score += 1

    if paper.get("source") == "openalex":
        score += 1

    levels = {
        4:"High",
        3:"Moderate",
        2:"Medium",
        1:"Low",
        0:"Very Low"
    }

    return levels[score]

## Create Processing Agent

In [43]:
def process_paper(
    paper,
    research_question
):
    raw_summary = paper.get("summary", "")

    # Bypass LLM summarization if abstract is missing or a generic placeholder
    if not raw_summary or "abstract not available" in raw_summary.lower():
        summary = f"No abstract available. Title: {paper.get('title', '')}"
    else:
        summary = summarize_paper(raw_summary)

    keywords = extract_keywords(
        summary
    )

    evaluation = evaluate_paper(
        paper,
        research_question
    )

    citation_check = (
        verify_citation(
            paper
        )
    )

    evidence = (
        assess_evidence_quality(
            paper
        )
    )

    paper["processed_summary"] = summary
    paper["keywords"] = keywords
    paper["relevance_score"] = evaluation["score"]
    paper["score_reason"] = evaluation["reason"]
    paper["citation_check"] = citation_check
    paper["evidence_quality"] = evidence

    return paper

In [44]:
# Re-process all retrieved papers with our new robust process_paper function
processed_clean = []
for paper in state.retrieved_papers:
    processed_clean.append(process_paper(paper, state.question))

state.processed_papers = processed_clean

# Re-sort by relevance score
state.processed_papers.sort(key=lambda x: x["relevance_score"], reverse=True)

In [51]:
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer
from uuid import uuid4
from datetime import datetime

# Fallback definition of ResearchMemory if not run yet
try:
    memory = ResearchMemory()
except NameError:
    class ResearchMemory:
        def __init__(self, db_path="./research_memory"):
            self.client = PersistentClient(path=db_path)
            self.collection = self.client.get_or_create_collection(name="academic_research")
            self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

        def store_paper(self, title, summary, source, url, research_question):
            document = f"Title: {title}\n\nSummary:\n{summary}\n\nSource:\n{source}\n\nURL:\n{url}"
            embedding = self.embedding_model.encode(document).tolist()
            paper_id = str(uuid4())
            self.collection.add(
                ids=[paper_id],
                documents=[document],
                embeddings=[embedding],
                metadatas=[{
                    "title": title,
                    "source": source,
                    "url": url,
                    "research_question": research_question,
                    "timestamp": datetime.now().isoformat()
                }]
            )
            return paper_id

        def search(self, query, top_k=5):
            query_embedding = self.embedding_model.encode(query).tolist()
            results = self.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k
            )
            return results
    memory = ResearchMemory()

# Clear existing database entries and store clean summaries
try:
    memory.client.delete_collection("academic_research")
except Exception:
    pass

# Re-get the collection cleanly
memory.collection = memory.client.get_or_create_collection(name="academic_research")

for paper in state.processed_papers:
    memory.store_paper(
        title=paper.get("title", ""),
        summary=paper.get("processed_summary", ""),
        source=paper.get("source", ""),
        url=paper.get("url", ""),
        research_question=state.question
    )

# Test retrieval on clean data
results = memory.search("The impact of modern technology on student learning experiences in higher education in europe")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("TITLE:", meta.get("title"))
    print(doc[:300])
    print("-" * 50)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TITLE: Innovative technologies in modern higher education: European experience and Ukrainian context
Title: Innovative technologies in modern higher education: European experience and Ukrainian context

Summary:
No abstract available. Title: Innovative technologies in modern higher education: European experience and Ukrainian context

Source:
openalex

URL:
https://openalex.org/W3207323781
--------------------------------------------------
TITLE: The practices and challenges of telecollaboration in higher education in Europe
Title: The practices and challenges of telecollaboration in higher education in Europe

Summary:
No abstract available. Title: The practices and challenges of telecollaboration in higher education in Europe

Source:
openalex

URL:
https://openalex.org/W2183802543
--------------------------------------------------
TITLE: Effects of traditional, blended and e‐learning on students' achievement in higher education
Title: Effects of traditional, blended and e‐learning o

## Process All Papers

In [52]:
processed = []

for paper in state.retrieved_papers:

    processed.append(

        process_paper(
            paper,
            state.question
        )
    )

state.processed_papers = processed


Rank Papers

In [53]:
state.processed_papers.sort(

    key=lambda x:
    x["relevance_score"],

    reverse=True
)

View Top Results

In [54]:
for paper in state.processed_papers[:5]:

    print(
        "\nTITLE:",
        paper["title"]
    )

    print(
        "SCORE:",
        paper["relevance_score"]
    )

    print(
        "QUALITY:",
        paper["evidence_quality"]
    )

    print(
        "REASON:",
        paper["score_reason"]
    )

    print("-" * 50)


TITLE: Exploring the Potential Impact of Artificial Intelligence (AI) on International Students in Higher Education: Generative AI, Chatbots, Analytics, and International Student Success
SCORE: 0.443
QUALITY: High
REASON: Similarity=0.28, Recency=0.70, Citations=0.46
--------------------------------------------------

TITLE: Exploring the impact of artificial intelligence on teaching and learning in higher education
SCORE: 0.432
QUALITY: Moderate
REASON: Similarity=0.40, Recency=0.10, Citations=1.00
--------------------------------------------------

TITLE: Impact of online tools on the learning experience of students in higher education
SCORE: 0.431
QUALITY: Moderate
REASON: Similarity=0.44, Recency=0.70, Citations=0.01
--------------------------------------------------

TITLE: Investigation of student experiences with ChatGPT-supported online learning applications in higher education
SCORE: 0.397
QUALITY: High
REASON: Similarity=0.34, Recency=0.70, Citations=0.07
-------------------

## Research Gap Agent

Use the LLM to identify trends and gaps across retrieved literature.

In [85]:
def identify_research_gaps(
    papers
):

    corpus = "\n\n".join(

        p["processed_summary"]

        for p in papers[:10]
    )

    response = (
        client.chat.completions.create(
            model=
            "openai/gpt-oss-120b",

            messages=[
                {
                    "role":"system",
                    "content":
                    """
                    Review the literature.

                    Identify:

                    1. Common themes
                    2. Research gaps
                    3. Future work

                    Return in markdown.
                    """
                },

                {
                    "role":"user",
                    "content": corpus
                }
            ]
        )
    )

    return (

        response
        .choices[0]
        .message.content
    )

Generate Gap Analysis

In [86]:
gap_analysis_result = identify_research_gaps(
    state.processed_papers
)

# Bypassing Pydantic's strict assignment restrictions
state.__dict__["gap_analysis"] = gap_analysis_result

print(state.__dict__["gap_analysis"])

## Literature Review: How AI Predictions Shape Human Decision‑Making  

### 1. Common Themes in the Existing Research  

| Theme | Typical Findings | Representative Studies |
|-------|------------------|------------------------|
| **Algorithmic Authority & Trust** | People tend to treat AI outputs as authoritative, even when performance is poor. Trust is often driven more by perceived expertise than by observed accuracy. | Lee & See (2004); **Liu et al. (2022)** – “The Illusion of Predictive Power.” |
| **Automation Bias & Disuse** | Users either over‑rely on AI recommendations (automation bias) or ignore them when they appear unreliable (automation disuse). The balance shifts with confidence cues and framing. | **Parasuraman & Manzey (2010)**; **Schoemaker et al. (2021)** – “When Trust Becomes Blind.” |
| **Framing & Decision Context** | The way AI predictions are presented (textual, visual, confidence scores) influences the extent of compliance. High‑stakes or loss‑aversion framing a

In [87]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

### Create Persistent ChromaDB Memory

In [107]:
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer
from uuid import uuid4
from datetime import datetime

class ResearchMemory:
    def __init__(self, db_path="./research_memory"):
        self.client = PersistentClient(path=db_path)
        self.collection = self.client.get_or_create_collection(name="academic_research")
        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    def store_paper(self, title, summary, source, url, research_question):
        document = f"""Title: {title}

Summary:
{summary}

Source:
{source}

URL:
{url}"""
        embedding = self.embedding_model.encode(document).tolist()
        paper_id = str(uuid4())
        self.collection.add(
            ids=[paper_id],
            documents=[document],
            embeddings=[embedding],
            metadatas=[{
                "title": title,
                "source": source,
                "url": url,
                "research_question": research_question,
                "timestamp": datetime.now().isoformat()
            }]
        )
        return paper_id

    def store_research_run(self, question, final_report):
        embedding = self.embedding_model.encode(final_report).tolist()
        run_id = str(uuid4())
        self.collection.add(
            ids=[run_id],
            documents=[final_report],
            embeddings=[embedding],
            metadatas=[{
                "type": "research_report",
                "question": question,
                "timestamp": datetime.now().isoformat()
            }]
        )
        return run_id

    def search(self, query, top_k=5):
        query_embedding = self.embedding_model.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        return results

Store Research Results

In [108]:
    def store_research_run(
        self,
        question,
        final_report
    ):

        embedding = self.embedding_model.encode(
            final_report
        ).tolist()

        run_id = str(uuid4())

        self.collection.add(
            ids=[run_id],
            documents=[final_report],
            embeddings=[embedding],
            metadatas=[{
                "type": "research_report",
                "question": question,
                "timestamp": datetime.now().isoformat()
            }]
        )

        return run_id

Semantic Search

Add semantic retrieval capability.

In [109]:
    def search(
        self,
        query,
        top_k=5
    ):

        query_embedding = self.embedding_model.encode(
            query
        ).tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        return results

Retrieve Previous Research

In [110]:
memory = ResearchMemory()

# Let's populate the database first with our processed papers so we have something to search!
for paper in state.processed_papers:
    memory.store_paper(
        title=paper.get("title", ""),
        summary=paper.get("processed_summary", ""),
        source=paper.get("source", ""),
        url=paper.get("url", ""),
        research_question=state.question
    )

results = memory.search(
    "applications of explainable AI in healthcare"
)

for doc, meta in zip(
    results["documents"][0],
    results["metadatas"][0]
):
    print("TITLE:", meta.get("title"))
    print(doc[:300])
    print("-" * 50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TITLE: Real‐World Applications of Explainable AI in Healthcare
Title: Real‐World Applications of Explainable AI in Healthcare

Summary:
No abstract available. Title: Real‐World Applications of Explainable AI in Healthcare

Source:
openalex

URL:
https://openalex.org/W4408383940
--------------------------------------------------
TITLE: Real‐World Applications of Explainable AI in Healthcare
Title: Real‐World Applications of Explainable AI in Healthcare

Summary:
No abstract available. Title: Real‐World Applications of Explainable AI in Healthcare

Source:
openalex

URL:
https://openalex.org/W4408383940
--------------------------------------------------
TITLE: Real-World Applications of Explainable AI in Healthcare
Title: Real-World Applications of Explainable AI in Healthcare

Summary:
No abstract available. Title: Real-World Applications of Explainable AI in Healthcare

Source:
openalex

URL:
https://openalex.org/W4396580664
--------------------------------------------------
TITLE: Rea

Connect to Your Processing Agent

In [55]:
research_q = state.question if 'state' in globals() else "Applications of Explainable AI in Healthcare"

memory.store_paper(
    title=paper["title"],
    summary=paper["summary"],
    source=paper["source"],
    url=paper["url"],
    research_question=research_q
)

'175c2e6b-8b16-4c4d-ad02-6cbb0ab4d0b7'

## Store Final Research Report

In [57]:
# Resolve NameError by mapping to existing active variables
active_question = state.question if 'state' in globals() else research_q
report_content = gap_analysis_result if 'gap_analysis_result' in globals() else "No report generated."

if hasattr(memory, 'store_research_run'):
    memory.store_research_run(
        question=active_question,
        final_report=report_content
    )
else:
    # Fallback storage directly into the collection if the custom method is missing
    from uuid import uuid4
    from datetime import datetime
    embedding = memory.embedding_model.encode(report_content).tolist()
    run_id = str(uuid4())
    memory.collection.add(
        ids=[run_id],
        documents=[report_content],
        embeddings=[embedding],
        metadatas=[{
            "type": "research_report",
            "question": active_question,
            "timestamp": datetime.now().isoformat()
        }]
    )

In [58]:
# Resolve NameError by using the active question variable
query_question = research_q if 'research_q' in globals() else "Applications of Explainable AI in Healthcare"

memory_results = memory.search(
    query_question,
    top_k=3
)

memory_context = ""

# Corrected loop syntax and nested ChromaDB results access
for doc in memory_results["documents"][0]:
    memory_context += doc + "\n\n"

planner_prompt = f"""
Previous Research:

{memory_context}

New Question:

{query_question}

Generate a research plan.
"""

Create Retrieval Function

In [59]:
def retrieve_context(
    self,
    query: str,
    top_k: int = 5
):

    query_embedding = self.embedding_model.encode(
        query
    ).tolist()

    results = self.collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    context = []

    if results["documents"]:

        for doc in results["documents"][0]:
            context.append(doc)

    return "\n\n".join(context)

## Create RAG Service

In [60]:
%%writefile rag.py
from memory import ResearchMemory

memory = ResearchMemory()


def build_rag_context(
    research_question
):

    context = memory.retrieve_context(
        research_question,
        top_k=5
    )

    return context

Writing rag.py


Create UI Function

In [64]:
def run_research_interface(
    research_question,
    selected_sources,
    max_cycles
):

    if not research_question.strip():

        return (
            " Please enter a research question",
            "",
            "",
            "",
            None,
            None
        )

    try:

        state = run_research(
            research_question=research_question,
            sources=selected_sources,
            max_cycles=max_cycles
        )

        # Research Plan
        plan_text = "# Research Plan\n\n"

        for task in state.plan["tasks"]:

            plan_text += (
                f"• {task['query']}\n"
                f"  - {task['purpose']}\n\n"
            )

        # Papers
        papers_text = ""

        for i, paper in enumerate(
            state.processed_papers[:10],
            start=1
        ):

            papers_text += f"""
### {i}. {paper['title']}

**Source:** {paper.get('source')}

**Year:** {paper.get('year')}

**Score:** {paper.get('relevance_score')}

**Evidence Quality:** {paper.get('evidence_quality')}

**Keywords:** {', '.join(paper.get('keywords', []))}

**Reasoning:** {paper.get('score_reason')}

---
"""

        gap_text = state.gap_analysis

        status = f"""
## Research Completed

**Question:** {state.question}

**Papers Retrieved:** {len(state.retrieved_papers)}

**Papers Processed:** {len(state.processed_papers)}

**Cycles:** {state.cycle}

**Sufficient Literature:** {state.sufficient}
"""

        markdown_file = generate_markdown_report(
            state
        )

        pdf_file = generate_pdf_report(
            state
        )

        return (
            status,
            plan_text,
            papers_text,
            gap_text,
            markdown_file,
            pdf_file
        )

    except Exception as e:

        return (
            f" Error: {str(e)}",
            "",
            "",
            "",
            None,
            None
        )

In [65]:
from pathlib import Path

def generate_markdown_report(state):

    output_dir = Path(
        "/content/research_output"
    )

    output_dir.mkdir(
        exist_ok=True
    )

    filename = (
        output_dir /
        "research_report.md"
    )

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"# Research Report\n\n"
        )

        f.write(
            f"## Question\n\n"
        )

        f.write(
            f"{state.question}\n\n"
        )

        f.write(
            "## Top Papers\n\n"
        )

        for paper in state.processed_papers:

            f.write(
                f"### {paper['title']}\n\n"
            )

            f.write(
                f"Score: {paper['relevance_score']}\n\n"
            )

            f.write(
                f"{paper['processed_summary']}\n\n"
            )

    return str(filename)

In [66]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import (
    getSampleStyleSheet
)

def generate_pdf_report(state):

    output_dir = Path(
        "/content/research_output"
    )

    output_dir.mkdir(
        exist_ok=True
    )

    pdf_path = (
        output_dir /
        "research_report.pdf"
    )

    doc = SimpleDocTemplate(
        str(pdf_path)
    )

    styles = (
        getSampleStyleSheet()
    )

    elements = []

    elements.append(
        Paragraph(
            "Research Report",
            styles["Title"]
        )
    )

    elements.append(
        Spacer(1,12)
    )

    elements.append(
        Paragraph(
            state.question,
            styles["BodyText"]
        )
    )

    for paper in state.processed_papers[:10]:

        elements.append(
            Paragraph(
                paper["title"],
                styles["Heading2"]
            )
        )

        elements.append(
            Paragraph(
                paper["processed_summary"],
                styles["BodyText"]
            )
        )

        elements.append(
            Spacer(1,10)
        )

    doc.build(elements)

    return str(pdf_path)

In [67]:
research_history = []

In [68]:
def save_history(question):

    research_history.append(
        question
    )

    return "\n".join(
        research_history
    )

In [72]:
import json
import gradio as gr

# Helper to dynamically list research questions stored in research_history
def list_history():
    return research_history

def export_reports(state_data):
    # Placeholders for report files since we handle PDF and MD generation in separate notebook cells
    return "/content/research_output/research_report.md", "/content/research_output/research_report.pdf"

def _paper_table(state):
    # Robust parsing in case state.retrieved_papers is used
    papers_to_use = getattr(state, 'processed_papers', getattr(state, 'retrieved_papers', []))
    return [[p.get("relevance_score", 0), p.get("year") or "", p.get("title", ""),
        ", ".join(p.get("authors") or [])[:180], p.get("source", ""),
        p.get("url", ""), p.get("summary", "")[:700]] for p in papers_to_use]

def research(question, limit, use_llm, email):
    if not question or not question.strip():
        raise gr.Error("Enter a research question first.")

    # Use the active notebook orchestrator run_research
    state = run_research(question, ["arxiv", "openalex"], max_cycles=1)
    save_history(question)

    plan_md = "\n".join(f"- **{x['purpose']}** — {x['query']}" for x in state.plan.get("tasks", [])) or "No plan created."
    status_md = f"- **Sufficient Literature**: {state.sufficient}\n- **Cycle**: {state.cycle}"

    findings = getattr(state, 'gap_analysis', "Literature synthesis in progress...")

    # Generate reports
    md_path = generate_markdown_report(state)
    pdf_path = generate_pdf_report(state)

    return (f"Retrieved **{len(state.retrieved_papers)} unique papers**. History updated.",
        plan_md, status_md, findings, _paper_table(state), md_path, pdf_path,
        gr.update(choices=list_history(), value=None))

def load_history(filename):
    if not filename:
        return "Select a history file."
    return json.dumps({"question": filename, "status": "Stored in history"}, indent=2, ensure_ascii=False)

def build_app():
    with gr.Blocks(title="Academic Research Agent", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Academic Research Agent\nMulti-agent academic literature discovery, synthesis and report export.")
        question = gr.Textbox(label="Research question", placeholder="e.g. How effective are SVM and LSTM models for detecting COVID-19 misinformation?", lines=3)
        with gr.Row():
            limit = gr.Slider(3, 15, value=8, step=1, label="Results per source and query")
            use_llm = gr.Checkbox(value=True, label="Use Groq Llama for planning, summaries and synthesis")
        email = gr.Textbox(label="Optional email for OpenAlex polite pool", placeholder="name@university.edu")
        run = gr.Button("Start research", variant="primary")
        summary = gr.Markdown()
        with gr.Tab("Research plan"):
            plan = gr.Markdown()
        with gr.Tab("Agent status and warnings"):
            status = gr.Markdown()
        with gr.Tab("Literature synthesis"):
            findings = gr.Markdown()
        with gr.Tab("Ranked papers"):
            papers = gr.Dataframe(headers=["Score", "Year", "Title", "Authors", "Source", "URL", "Summary"],
                datatype=["number","str","str","str","str","str","str"], interactive=False, wrap=True)
        with gr.Row():
            md_file = gr.File(label="Download Markdown report")
            pdf_file = gr.File(label="Download PDF report")
        history_choice = gr.Dropdown(choices=list_history(), label="Saved research runs")
        load_btn = gr.Button("Load selected history")
        history_json = gr.Code(label="History record (JSON)", language="json")

        run.click(research, inputs=[question, limit, use_llm, email],
            outputs=[summary, plan, status, findings, papers, md_file, pdf_file, history_choice])
        load_btn.click(load_history, inputs=[history_choice], outputs=[history_json])
    return demo

In [ ]:
app = build_app()
app.launch(
    share=True,
    debug=True,
    show_error=True
)

/tmp/ipykernel_788/3918705964.py:46: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Academic Research Agent", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://92deea0973f634c6bd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
